In [1]:
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

### Feature Engineering

In [4]:
# Name -> Title -> Few Rare Title
train['Title'] = train['Name'].str.extract(r',\s*([^\.]*)\.')
test['Title']  = test['Name'].str.extract(r',\s*([^\.]*)\.')

rare = train['Title'].value_counts()[train['Title'].value_counts() < 10].index
train['Title'] = train['Title'].replace(rare, 'Rare').replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
test['Title']  = test['Title'].replace(rare, 'Rare').replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
test['Title']  = test['Title'].where(test['Title'].isin(train['Title'].unique()), 'Rare')

# Family Size = SibSp + ParCh
train['Family_Size'] = train['SibSp'] + train['Parch']
test['Family_Size'] = test['SibSp'] + test['Parch']
train['IsAlone'] = (train['Family_Size'] == 0).astype(int)
test['IsAlone']  = (test['Family_Size'] == 0).astype(int)

# Cabin Imputation
train['Cabin'] = train['Cabin'].fillna('Missing')
test['Cabin'] = test['Cabin'].fillna('Missing')

# Cabin Code Extract
train['Deck'] = train['Cabin'].astype(str).str[0]
test['Deck'] = test['Cabin'].astype(str).str[0]

# Ticket Group
ticket_counts = pd.concat([train['Ticket'], test['Ticket']]).value_counts()
train['Ticket_Group'] = train['Ticket'].map(ticket_counts)
test['Ticket_Group']  = test['Ticket'].map(ticket_counts)

In [5]:
# Instead of imputing missing ages, predicting it using Title, Pclass, SibSp, Parch, Fare
def fill_age(df_train, df_apply):
    features = ['Pclass', 'SibSp', 'Parch', 'Fare']
    title_dummies_train = pd.get_dummies(df_train['Title'], prefix='Title')
    title_dummies_apply = pd.get_dummies(df_apply['Title'], prefix='Title')
    title_dummies_apply = title_dummies_apply.reindex(columns=title_dummies_train.columns, fill_value=0)

    X_age_train = pd.concat([df_train[features], title_dummies_train], axis=1)
    X_age_apply = pd.concat([df_apply[features], title_dummies_apply], axis=1)
    X_age_apply = X_age_apply.fillna(X_age_train.median())
    X_age_train_full = X_age_train.fillna(X_age_train.median())

    known_age = df_train['Age'].notna()
    age_model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
    age_model.fit(X_age_train_full[known_age], df_train.loc[known_age, 'Age'])

    missing_rows = df_apply['Age'].isna()
    df_apply.loc[missing_rows, 'Age'] = age_model.predict(X_age_apply[missing_rows])
    return df_apply

train = fill_age(train, train)
test  = fill_age(train, test) 

### Train_Test Split

In [6]:
X = train.drop('Survived', axis=1)
y = train['Survived']

In [7]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Data Preprocessing Pipeline

In [8]:
p1 = Pipeline(
    steps=[
        ('scaler', StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]
)

p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
    ]
)

In [9]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Family_Size,IsAlone,Deck,Ticket_Group
692,693,3,"Lam, Mr. Ali",male,28.469497,0,0,1601,56.4958,Missing,S,Mr,0,1,M,8
481,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,34.239887,0,0,239854,0.0000,Missing,S,Mr,0,1,M,1
527,528,1,"Farthing, Mr. John",male,32.208020,0,0,PC 17483,221.7792,C95,S,Mr,0,1,C,4
855,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.000000,0,1,392091,9.3500,Missing,S,Mrs,1,0,M,2
801,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.000000,1,1,C.A. 31921,26.2500,Missing,S,Mrs,2,0,M,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,3,"Mockler, Miss. Helen Mary ""Ellie""",female,23.174962,0,0,330980,7.8792,Missing,Q,Miss,0,1,M,1
258,259,1,"Ward, Miss. Anna",female,35.000000,0,0,PC 17755,512.3292,Missing,C,Miss,0,1,M,4
736,737,3,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.000000,1,3,W./C. 6608,34.3750,Missing,S,Mrs,4,0,M,5
462,463,1,"Gee, Mr. Arthur H",male,47.000000,0,0,111320,38.5000,E63,S,Mr,0,1,E,1


In [10]:
y_train

692    1
481    0
527    0
855    1
801    1
      ..
359    1
258    1
736    0
462    0
507    1
Name: Survived, Length: 712, dtype: int64

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline1', p1, ['Age']),
        ('pipeline2', p2, ['Fare', 'Family_Size', 'Ticket_Group']),
        ('pipeline3', p3, ['Embarked', 'Sex', 'Deck', 'Title', 'IsAlone', 'Pclass']),
    ],
    remainder='drop'
)
preprocessor

,transformers,"[('pipeline1', ...), ('pipeline2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


### Model Training

In [ ]:
# Hyperparameter for GradientBoosting 
gb_params = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [2, 3, 4, 5],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
}

gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gb_search = RandomizedSearchCV(
    gb_pipe,
    gb_params,
    n_iter=40,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    refit=True  
)

gb_search.fit(X_train, y_train)
model = gb_search.best_estimator_

### Model Prediction

In [13]:
y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy: {accuracy}")

precision = precision_score(y_val, y_pred)
print(f"Precision: {precision}")

recall = recall_score(y_val, y_pred)
print(f"Recall: {recall}")


Accuracy: 0.7932960893854749
Precision: 0.7666666666666667
Recall: 0.6666666666666666


In [14]:
print('CV score on X_train folds:', gb_search.best_score_)

CV score on X_train folds: 0.8371318822023047


In [15]:
y_test_pred = model.predict(test)

In [16]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': y_test_pred
})

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [17]:
submission.to_csv('submission.csv', index=False)